# Policy Rendering for Trained ReachBot Models

This notebook loads a trained policy and renders videos of the robot's performance in the environment.

## Import Libraries and Setup

In [1]:
import mujoco
import os
import sys
import imageio

# Add the project root to the Python path to resolve module imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import jax
from etils import epath
import functools

from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.io import model
from jax import numpy as jp

from mujoco_playground import wrapper
from mujoco_playground.config import locomotion_params
from ml_collections import config_dict
import json

# Configure JAX for debugging
#jax.config.update("jax_debug_infs", True)

print("Libraries imported successfully!")

Libraries imported successfully!


## Configuration and Path Setup

In [2]:
# Set up paths
script_dir = os.path.dirname(os.path.abspath(__file__))
relative_ckpt_path = "cave_exploration/logs/cave_exploration-2025-07-15_09-13-47"
ckpt_path = os.path.join(script_dir, relative_ckpt_path)

print(f"Script directory: {script_dir}")
print(f"Checkpoint path: {ckpt_path}")
print(f"Checkpoint exists: {os.path.exists(ckpt_path)}")

Script directory: /home/ga53voq/master_thesis/tasks
Checkpoint path: /home/ga53voq/master_thesis/tasks/cave_exploration/logs/cave_exploration-2025-07-15_09-13-47
Checkpoint exists: True


## Load Configuration and Determine Task Type

In [3]:
# Load the configuration from the checkpoint
with open(os.path.join(ckpt_path, 'config.json'), 'r') as f:
    loaded_config = json.load(f)

# Determine task type and get the appropriate configuration
if 'joystick' in relative_ckpt_path:
    print('Rendering joystick task result')
    # Uncomment these lines when needed
    # from reachbot.joystick import default_config as reachbot_joystick_config
    # env_cfg = reachbot_joystick_config()
elif 'getup' in relative_ckpt_path:
    print('Rendering getup task result')
    # Uncomment these lines when needed
    # from reachbot.getup import default_config as reachbot_getup_config
    # env_cfg = reachbot_getup_config()
elif 'cave_exploration' in relative_ckpt_path:
    print('Rendering cave exploration task result')
    from tasks.cave_exploration.cave_exploration import default_config as cave_exploration_config
    env_cfg = cave_exploration_config()
else:
    print('Unknown task')
    raise ValueError("Unknown task type in checkpoint path")

# Convert the loaded dict to a ConfigDict and update the default config
json_env_cfg = config_dict.ConfigDict(loaded_config['env_cfg'])
env_cfg.update(json_env_cfg)

print("Configuration loaded successfully!")

Rendering cave exploration task result
Configuration loaded successfully!


## Initialize Environment

In [4]:
# Create the appropriate environment based on task type
if 'joystick' in relative_ckpt_path:
    # Uncomment these lines when needed
    # from reachbot.joystick import Joystick as ReachbotJoystick
    # env = ReachbotJoystick(config=env_cfg, task="rough_terrain_basic")
    pass
elif 'getup' in relative_ckpt_path:
    # Uncomment these lines when needed
    # from reachbot.getup import Getup as ReachbotGetup
    # env = ReachbotGetup(config=env_cfg, task="flat_terrain_basic")
    pass
elif 'cave_exploration' in relative_ckpt_path:
    from tasks.cave_exploration.cave_exploration import CaveExplore
    env = CaveExplore(config=env_cfg)
    
    # Print actuator information
    print("MuJoCo model actuator control ranges:")
    print(f"  Control range: {env.mj_model.actuator_ctrlrange}")
    print(f"  Control limited: {env.mj_model.actuator_ctrllimited}")
    print("\nMJX model actuator control ranges:")
    print(f"  Control range: {env.mjx_model.actuator_ctrlrange}")
    print(f"  Control limited: {env.mjx_model.actuator_ctrllimited}")

print("Environment initialized successfully!")

Found 3 cave folders in /home/ga53voq/master_thesis/tasks/cave_exploration/environment/caves.
Loading 1 cave environments.
initial_qpos (cave_batch_loader): [ 0.     0.033  0.054  0.994 -0.179  0.    -0.     0.    -0.004 -0.002
  0.001  0.002  0.006 -0.    -0.001 -0.001 -0.001  0.002  0.006]
Floor boxes detected: 4025
initial_qpos (cave_batch_loader): [ 0.     0.033  0.054  0.994 -0.179  0.    -0.     0.    -0.004 -0.002
  0.001  0.002  0.006 -0.    -0.001 -0.001 -0.001  0.002  0.006]
Floor boxes detected: 4025
Found 4 boom end geoms
Found 4026 floor/wall geoms
CaveExplore task initialized with model: ReachbotModelType.BASIC
CaveExplore task action space: 12
Found 4 boom end geoms
Found 4026 floor/wall geoms
CaveExplore task initialized with model: ReachbotModelType.BASIC
CaveExplore task action space: 12
CaveExplore task observation space: {'privileged_state': (164,), 'state': (105,)}
MuJoCo model actuator control ranges:
  Control range: [[-3.142  3.142]
 [ 0.     4.014]
 [ 0.     1.

## Setup PPO Configuration and Training Function

In [5]:
# Get the PPO configuration
ppo_params = locomotion_params.brax_ppo_config('Go1JoystickFlatTerrain')
ppo_training_params = dict(ppo_params)
ppo_training_params['num_timesteps'] = 0
ppo_training_params['num_envs'] = 2

# Setup network factory
if "network_factory" in ppo_params:
    if "network_factory" in ppo_training_params:
        del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks,
        **ppo_params.network_factory
    )

# Build the training function
train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
)

print("PPO configuration setup complete!")

PPO configuration setup complete!


## Load Trained Model and Setup Inference

In [6]:
# Build the inference function
make_inference_fn, params, _ = train_fn(
    environment=env,
    num_timesteps=0,
    wrap_env_fn=wrapper.wrap_for_brax_training
)

# Load the trained model parameters
params = model.load_params(os.path.join(ckpt_path, 'params'))

# JIT compile functions for faster execution
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

print("Model loaded and functions compiled successfully!")

/home/ga53voq/.conda/envs/pyenv/lib/python3.12/site-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))


/home/ga53voq/.conda/envs/pyenv/lib/python3.12/site-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))


Model loaded and functions compiled successfully!


## Rollout Configuration

In [7]:
# Rollout parameters
rng = jax.random.PRNGKey(3)
rollout = []
n_episodes = 3
episode_length = 5000

print(f"Rollout configuration:")
print(f"  Number of episodes: {n_episodes}")
print(f"  Episode length: {episode_length} steps")
print(f"  Total steps: {n_episodes * episode_length}")

Rollout configuration:
  Number of episodes: 3
  Episode length: 5000 steps
  Total steps: 15000


## Run Policy Rollout

In [ ]:
# Rollout policy and record simulation
print(f"Running rollout for {n_episodes} episode(s) with {episode_length} steps each...")

for episode in range(n_episodes):
    episode_reward = 0.0
    print(f"\nEpisode {episode + 1}/{n_episodes}")
    
    state = jit_reset(rng)
    rollout.append(state)
    
    for i in range(episode_length):
        if i % 500 == 0:
            print(f"  Step {i}/{episode_length}")
            
        act_rng, rng = jax.random.split(rng)
        ctrl, _ = jit_inference_fn(state.obs, act_rng)
        
        # Check for numerical issues
        if jp.any(jp.isinf(ctrl)) or jp.any(jp.isnan(ctrl)):
            print(f"Numerical issue detected in control at step {i}. Stopping rollout.")
            break
            
        state = jit_step(state, ctrl)

        # Accumulate reward for this episode
        episode_reward += float(state.reward)
        
        if state.done:
            print(f"Episode {episode + 1} ended at step {i} with reward: {episode_reward:.3f}")
            break
            
        rollout.append(state)

    # Render video
    print("Rendering video...")

    # Rendering parameters
    render_every = 1    # Render every frame
    width = 1920        # Full HD width
    height = 1080       # Full HD height

    frames = env.render(rollout[::render_every], camera='track_global', width=width, height=height)
    print(f"Rendered {len(frames)} frames")

    # Save video
    video_path = os.path.join(relative_ckpt_path, f'posttraining_{episode_reward:.2f}.mp4')
    fps = 1.0 / env.dt

    print(f"Saving video to {video_path} at {fps} FPS...")
    imageio.mimsave(video_path, frames, fps=fps)
    print(f"Video saved successfully to {video_path}")


Running rollout for 3 episode(s) with 5000 steps each...

Episode 1/3


Running rollout for 3 episode(s) with 5000 steps each...

Episode 1/3


2025-07-15 13:42:39.550477: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


Running rollout for 3 episode(s) with 5000 steps each...

Episode 1/3


2025-07-15 13:42:39.550477: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


...

Running rollout for 3 episode(s) with 5000 steps each...

Episode 1/3


2025-07-15 13:42:39.550477: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


  Step 0/5000
Max contacts updated to 256 based on current step.
Max contacts updated to 256 based on current step.
  Step 500/5000
  Step 500/5000
  Step 1000/5000
  Step 1000/5000
Episode 1 ended at step 1251 with reward: -2.399
Rendering video...
Episode 1 ended at step 1251 with reward: -2.399
Rendering video...


100%|██████████| 1252/1252 [15:07<00:00,  1.38it/s]

/home/ga53voq/.conda/envs/pyenv/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(
/home/ga53voq/.conda/envs/pyenv/lib/python3.12/subprocess.py:1885: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _fork_exec(


Rendered 1252 frames
Saving video to cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/posttraining_-2.40.mp4 at 50.0 FPS...
Video saved successfully to cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/posttraining_-2.40.mp4

Episode 2/3
  Step 0/5000
Video saved successfully to cave_exploration/logs/cave_exploration-2025-07-15_09-13-47/posttraining_-2.40.mp4

Episode 2/3
  Step 0/5000
  Step 500/5000
  Step 500/5000


## Summary

The policy rendering is complete! The notebook has:

1. Loaded the trained policy from the specified checkpoint
2. Run multiple episodes with the trained policy
3. Recorded the robot's performance
4. Rendered and saved a high-quality video of the results

Check the generated video file to see how well your trained model performs!